<a href="https://colab.research.google.com/github/JozefSL/pyNotes/blob/main/numpy/prodMatrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

def generate_base_schedule_matrix(type_curve, timeline_months):
    """
    Generates a reusable base schedule matrix where each row represents a well vintage
    (the month a batch of wells is turned online) and each column represents a calendar timeline month.

    Parameters:
    type_curve (np.ndarray): The absolute production profile vector for a single standard well.
    timeline_months (int): Total number of months to model in the forecast timeline.

    Returns:
    np.ndarray: A 2D matrix of shape (timeline_months, timeline_months) filled with shifted type curves.
    """
    base_matrix = np.zeros((timeline_months, timeline_months))

    for start_month in range(timeline_months):
        # Calculate how many months are left in the forecast timeline from this start month forward
        months_remaining = timeline_months - start_month

        # Determine how much of the type curve fits into the remaining timeline window
        curve_length = min(len(type_curve), months_remaining)

        # Populate the row starting from the diagonal (the month the well goes online)
        base_matrix[start_month, start_month:start_month + curve_length] = type_curve[:curve_length]

    return base_matrix

def run_scenario(base_matrix, well_count_vector):
    """
    Applies a specific scenario's well count vector to the base schedule matrix using
    NumPy broadcasting, and computes both detailed and collapsed timeline production.

    Parameters:
    base_matrix (np.ndarray): The unit production matrix (rows=vintage, cols=timeline month).
    well_count_vector (np.ndarray): 1D array representing the number of wells added each month.

    Returns:
    tuple: (detailed_production_matrix, total_timeline_production)
    """
    # Convert the flat 1D well count vector into a 2D column vector (shape: N x 1)
    # This enables NumPy broadcasting to multiply each row of the base matrix by its corresponding well count
    well_count_column = well_count_vector[:, np.newaxis]

    # Scale the base unit matrix by the scenario's well count column
    detailed_production_matrix = base_matrix * well_count_column

    # Sum vertically across the rows (axis=0) to collapse all active vintages into a single timeline forecast
    total_timeline_production = np.sum(detailed_production_matrix, axis=0)

    return detailed_production_matrix, total_timeline_production

if __name__ == "__main__":
    # 1. Setup Timeline Configuration
    forecast_duration_months = 12
    peak_production_rate = 500.0  # Peak rate in barrels of oil equivalent per day (BOE/d)

    # 2. Define a standard single-well normalized decline curve vector
    # Represents Month 1 at 100%, declining predictably over time
    normalized_decline = np.array([1.00, 0.85, 0.72, 0.61, 0.52, 0.44, 0.38, 0.32, 0.27, 0.23, 0.20, 0.17])
    single_well_profile = normalized_decline * peak_production_rate

    # 3. Generate the reusable Base Schedule Matrix template (completely independent of well counts)
    base_prod_matrix = generate_base_schedule_matrix(single_well_profile, forecast_duration_months)

    # 4. Define Varying Well-Count Vectors for Scenario Analysis
    # Scenario A: Steady Development Program (2 wells brought online every month)
    wells_scenario_steady = np.array([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

    # Scenario B: Back-loaded Program (Zero drilling first half, aggressive 4-well pace second half)
    wells_scenario_backloaded = np.array([0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4])

    # Scenario C: Pulsed/Campaign Program (5 wells brought online at the start of each quarter)
    wells_scenario_pulse = np.array([5, 0, 0, 5, 0, 0, 5, 0, 0, 5, 0, 0])

    # 5. Evaluate the Scenarios against the Base Template Matrix
    matrix_steady, total_steady = run_scenario(base_prod_matrix, wells_scenario_steady)
    matrix_backloaded, total_backloaded = run_scenario(base_prod_matrix, wells_scenario_backloaded)
    matrix_pulse, total_pulse = run_scenario(base_prod_matrix, wells_scenario_pulse)

    # 6. Display Comparison Reports
    print("=========================================================================")
    print("BASE PRODUCTION MATRIX TEMPLATE (Single Well Volume Shifting Over Time)")
    print("=========================================================================")
    print(np.round(base_prod_matrix, 0))
    print("\n")

    print("=========================================================================")
    print("SCENARIO ANALYSIS: TOTAL MONTHLY FIELD PRODUCTION FORECAST (BOE/d)")
    print("=========================================================================")
    print(f"Timeline Month | Steady Scenario | Back-loaded Scenario | Pulsed Scenario")
    print("-------------------------------------------------------------------------")
    for month_idx in range(forecast_duration_months):
        print(f"Month {month_idx + 1:<8} | {total_steady[month_idx]:<15,.0f} | {total_backloaded[month_idx]:<21,.0f} | {total_pulse[month_idx]:,.0f}")
    print("-------------------------------------------------------------------------")
    print(f"Cumulative Forecast Volume (BOE-Months):")
    print(f"  Steady Program Scenario:    {np.sum(total_steady):,.0f} BOE")
    print(f"  Back-loaded Program Scenario: {np.sum(total_backloaded):,.0f} BOE")
    print(f"  Pulsed Program Scenario:     {np.sum(total_pulse):,.0f} BOE")
    print("=========================================================================")


BASE PRODUCTION MATRIX TEMPLATE (Single Well Volume Shifting Over Time)
[[500. 425. 360. 305. 260. 220. 190. 160. 135. 115. 100.  85.]
 [  0. 500. 425. 360. 305. 260. 220. 190. 160. 135. 115. 100.]
 [  0.   0. 500. 425. 360. 305. 260. 220. 190. 160. 135. 115.]
 [  0.   0.   0. 500. 425. 360. 305. 260. 220. 190. 160. 135.]
 [  0.   0.   0.   0. 500. 425. 360. 305. 260. 220. 190. 160.]
 [  0.   0.   0.   0.   0. 500. 425. 360. 305. 260. 220. 190.]
 [  0.   0.   0.   0.   0.   0. 500. 425. 360. 305. 260. 220.]
 [  0.   0.   0.   0.   0.   0.   0. 500. 425. 360. 305. 260.]
 [  0.   0.   0.   0.   0.   0.   0.   0. 500. 425. 360. 305.]
 [  0.   0.   0.   0.   0.   0.   0.   0.   0. 500. 425. 360.]
 [  0.   0.   0.   0.   0.   0.   0.   0.   0.   0. 500. 425.]
 [  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0. 500.]]


SCENARIO ANALYSIS: TOTAL MONTHLY FIELD PRODUCTION FORECAST (BOE/d)
Timeline Month | Steady Scenario | Back-loaded Scenario | Pulsed Scenario
----------------------------

In [ ]:
import numpy as np

def generate_arps_profile(qi, di_annual, b, num_months):
    """Generates a monthly production profile for 1 well using Arps Hyperbolic Decline."""
    # Convert nominal annual decline rate to a monthly nominal rate
    di_monthly = di_annual / 12.0

    # Time steps: 0 for the first month, 1 for the second month, etc.
    t = np.arange(num_months)

    # Handle the special case where b = 0 (Exponential Decline)
    if b == 0:
        profile = qi * np.exp(-di_monthly * t)
    else:
        # Arps Hyperbolic equation
        profile = qi / ((1.0 + b * di_monthly * t) ** (1.0 / b))

    return profile

# 1. Setup Simulation Parameters
num_months = 12
initial_rate = 500      # qi: Initial barrels of oil equivalent per day (BOE/d)
annual_decline = 0.65   # di: 65% nominal initial decline per year
b_factor = 0.5          # b: Hyperbolic exponent

# 2. Generate the Single Well Production Vector using Arps
single_well_profile = generate_arps_profile(initial_rate, annual_decline, b_factor, num_months)

# 3. Build the Shifted Base Schedule Matrix
base_schedule_matrix = np.zeros((num_months, num_months))

for start_month in range(num_months):
    months_active = num_months - start_month
    # Place the Arps profile into the matrix starting at the well's online month
    base_schedule_matrix[start_month, start_month:] = single_well_profile[:months_active]

# Print Results
print("Single Well Arps Monthly Production Profile:")
print(np.round(single_well_profile, 1))

print("\nBase Schedule Matrix from Arps Coefficients (Rows = Start Month, Columns = Timeline Month):")
print(np.round(base_schedule_matrix, 1))


Single Well Arps Monthly Production Profile:
[500.  474.  449.9 427.7 407.  387.8 370.  353.3 337.8 323.2 309.6 296.8]

Base Schedule Matrix from Arps Coefficients (Rows = Start Month, Columns = Timeline Month):
[[500.  474.  449.9 427.7 407.  387.8 370.  353.3 337.8 323.2 309.6 296.8]
 [  0.  500.  474.  449.9 427.7 407.  387.8 370.  353.3 337.8 323.2 309.6]
 [  0.    0.  500.  474.  449.9 427.7 407.  387.8 370.  353.3 337.8 323.2]
 [  0.    0.    0.  500.  474.  449.9 427.7 407.  387.8 370.  353.3 337.8]
 [  0.    0.    0.    0.  500.  474.  449.9 427.7 407.  387.8 370.  353.3]
 [  0.    0.    0.    0.    0.  500.  474.  449.9 427.7 407.  387.8 370. ]
 [  0.    0.    0.    0.    0.    0.  500.  474.  449.9 427.7 407.  387.8]
 [  0.    0.    0.    0.    0.    0.    0.  500.  474.  449.9 427.7 407. ]
 [  0.    0.    0.    0.    0.    0.    0.    0.  500.  474.  449.9 427.7]
 [  0.    0.    0.    0.    0.    0.    0.    0.    0.  500.  474.  449.9]
 [  0.    0.    0.    0.    0.    0.  

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML

def generate_arps_profile(qi, di_annual, b, num_months):
    di_monthly = di_annual / 12.0
    t = np.arange(num_months)
    if b == 0:
        return qi * np.exp(-di_monthly * t)
    return qi / ((1.0 + b * di_monthly * t) ** (1.0 / b))

# 1. Load the CSV file
df = pd.read_csv("CurveData.txt")

# Identify our timeline length from the month columns (m1 through m12)
month_cols = [c for c in df.columns if c.startswith('m')]
num_months = len(month_cols)

# Dictionary to hold the final forecast timelines for analysis
regional_forecasts = {}

# 2. Process data dynamically by Region
for region, group in df.groupby('region'):

    # Extract Arps coefficients from the 'm1' column of coefficient rows
    qi = group.loc[group['name'] == 'qi', 'm1'].values[0]
    di_annual = group.loc[group['name'] == 'di_annual', 'm1'].values[0]
    b_factor = group.loc[group['name'] == 'b_factor', 'm1'].values[0]

    # Extract the 12-month drilling schedule vector
    well_count_vector = group.loc[group['name'] == 'well_count', month_cols].values[0]

    # Generate the single-well type curve
    single_well_profile = generate_arps_profile(qi, di_annual, b_factor, num_months)

    # Build the base schedule matrix for this specific region
    base_schedule_matrix = np.zeros((num_months, num_months))
    for start_month in range(num_months):
        months_active = num_months - start_month
        base_schedule_matrix[start_month, start_month:] = single_well_profile[:months_active]

    # Multiply the base matrix by the regional schedule vector
    # well_count_vector[:, np.newaxis] converts flat array to vertical column
    regional_prod_matrix = base_schedule_matrix * well_count_vector[:, np.newaxis]

    # Sum columns to get total timeline production for this region
    regional_forecasts[region] = np.sum(regional_prod_matrix, axis=0)

# 3. Combine results into a clean summary DataFrame
forecast_df = pd.DataFrame(regional_forecasts, index=[f"Month_{i+1}" for i in range(num_months)])
forecast_df['Total_Company'] = forecast_df.sum(axis=1)
forecast_df = forecast_df.astype(float)

print("Forecast Summary (BOE/day):")

display(forecast_df.style.format("{:,.0f}"))
#print(forecast_df)

Forecast Summary (BOE/day):


,Bakken,EagleFord,Permian,Total_Company
Month_1,0,500,"1,600","2,100"
Month_2,0,976,"3,111","4,086"
Month_3,"2,400","1,429","5,339","9,168"
Month_4,"4,675","1,861","7,447","13,984"
Month_5,"4,436","2,774","10,245","17,455"
Month_6,"4,216","3,644","12,895","20,755"
Month_7,"6,413","4,474","13,810","24,696"
Month_8,"8,500","5,266","14,688","28,455"
Month_9,"8,087","6,024","14,733","28,844"
Month_10,"7,705","6,748","14,789","29,243"
